In [18]:
import cv2
import numpy as np
import os
from skimage.feature import hog
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import joblib
import os
from tqdm import tqdm
import time
import pandas as pd


In [2]:
def load_images(folder, label):
    images = []
    labels = []
    files = [f for f in os.listdir(folder) if f.endswith('.png') or f.endswith('.jpg')]
    for filename in files:
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        images.append(img)
        labels.append(label)
    return images, labels

healthy_path = "/Users/nabiha/parkinsons_drawings/Healthy"
parkinson_path = "/Users/nabiha/parkinsons_drawings/Parkinson"

healthy_imgs, healthy_labels = load_images(healthy_path, 0)
parkinson_imgs, parkinson_labels = load_images(parkinson_path, 1)

all_images = healthy_imgs + parkinson_imgs
all_labels = healthy_labels + parkinson_labels

print("Total images:", len(all_images))
print("Class distribution — Healthy:", len(healthy_imgs), "Parkinson:", len(parkinson_imgs))

Total images: 3389
Class distribution — Healthy: 1757 Parkinson: 1632


In [3]:
def extract_hog_features(images):
    features = []
    for img in images:
        fd = hog(img, 
                 orientations=9, 
                 pixels_per_cell=(8, 8),
                 cells_per_block=(2, 2), 
                 block_norm='L2-Hys')
        features.append(fd)
    return np.array(features)

X = extract_hog_features(all_images)
y = np.array(all_labels)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (3389, 8100)


In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2711, 8100)
X_test shape: (678, 8100)


In [5]:
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("X_train_pca shape:", X_train_pca.shape)
print("X_test_pca shape:", X_test_pca.shape)
print("Variance explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

X_train_pca shape: (2711, 100)
X_test_pca shape: (678, 100)
Variance explained: 57.43 %


In [6]:
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Number of components selected:", pca.n_components_)
print("X_train_pca shape:", X_train_pca.shape)
print("Variance explained:", round(sum(pca.explained_variance_ratio_) * 100, 2), "%")

Number of components selected: 1055
X_train_pca shape: (2711, 1055)
Variance explained: 95.01 %


In [7]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    scoring='roc_auc', 
    cv=5,
    verbose=2
)

grid_search.fit(X_train_pca, y_train)

print("Best params:", grid_search.best_params_)
print("Best ROC-AUC:", grid_search.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.4s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=100; total time=   3.3s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.6s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   7.0s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.7s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.5s
[CV] END max_depth=None, min_samples_split=2, n_estimators=200; total time=   6.6s
[CV] END max_depth=None, min_samples_split=5, n_estimators=100; total time=   3.4s
[CV] END max_depth=None, mi

In [8]:
best_drawing_rf = grid_search.best_estimator_

y_pred = best_drawing_rf.predict(X_test_pca)
y_prob = best_drawing_rf.predict_proba(X_test_pca)[:, 1]

print("Drawing Model Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Drawing Model Results
Accuracy: 0.897

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.98      0.91       352
           1       0.98      0.80      0.88       326

    accuracy                           0.90       678
   macro avg       0.91      0.89      0.90       678
weighted avg       0.91      0.90      0.90       678

ROC AUC: 0.973

Confusion Matrix:
[[346   6]
 [ 64 262]]


In [9]:
joblib.dump(best_drawing_rf, '../models/drawing_model.pkl')
joblib.dump(pca, '../models/drawing_pca.pkl')

['../models/drawing_pca.pkl']

In [10]:
def fuse_predictions(clinical_prob, drawing_prob):
    diff = abs(clinical_prob - drawing_prob)
    
    if clinical_prob >= 0.65 and drawing_prob >= 0.65:
        return "High Risk — both models agree"
    elif clinical_prob <= 0.35 and drawing_prob <= 0.35:
        return "Low Risk — both models agree"
    elif diff > 0.3:
        return "Inconclusive — models disagree"
    else:
        return "Moderate Risk"

In [11]:

print(fuse_predictions(0.80, 0.75))  # should be High Risk
print(fuse_predictions(0.20, 0.25))  # should be Low Risk
print(fuse_predictions(0.80, 0.30))  # should be Inconclusive
print(fuse_predictions(0.55, 0.50))  # should be Moderate Risk

High Risk — both models agree
Low Risk — both models agree
Inconclusive — models disagree
Moderate Risk


In [12]:
models_dir = '../models/'
print("Files in models folder:")
for f in sorted(os.listdir(models_dir)):
    print(f"  {f}")


Files in models folder:
  X_test_scaled.pkl
  X_train_scaled.pkl
  class_distribution.csv
  clinical_model.pkl
  clinical_scaler.pkl
  confusion_matrix.csv
  drawing_model.pkl
  drawing_pca.pkl
  feature_importance.csv
  model_comparison.csv
  roc_clinical.csv
  selected_features.pkl
  shap_explainer.pkl
  shap_values.pkl
  y_test.pkl
  y_train.pkl


In [13]:
clinical_model = joblib.load('../models/clinical_model.pkl')
clinical_scaler = joblib.load('../models/clinical_scaler.pkl')
selected_features = joblib.load('../models/selected_features.pkl')
drawing_model = joblib.load('../models/drawing_model.pkl')
drawing_pca = joblib.load('../models/drawing_pca.pkl')
shap_explainer = joblib.load('../models/shap_explainer.pkl')

print("All models loaded successfully!")
print(f"Clinical model: {type(clinical_model).__name__}")
print(f"Drawing model: {type(drawing_model).__name__}")
print(f"Selected features ({len(selected_features)}):", selected_features)

All models loaded successfully!
Clinical model: RandomForestClassifier
Drawing model: RandomForestClassifier
Selected features (13): ['Age', 'DietQuality', 'TraumaticBrainInjury', 'Diabetes', 'Depression', 'UPDRS', 'MoCA', 'FunctionalAssessment', 'Tremor', 'Rigidity', 'Bradykinesia', 'PosturalInstability', 'SleepDisorders']


/opt/anaconda3/envs/parkinsons_jcp/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:

fusion_code = '''

def fuse_predictions(clinical_prob, drawing_prob):
    diff = abs(clinical_prob - drawing_prob)
    
    if clinical_prob >= 0.65 and drawing_prob >= 0.65:
        return "High Risk — both models agree"
    elif clinical_prob <= 0.35 and drawing_prob <= 0.35:
        return "Low Risk — both models agree"
    elif diff > 0.3:
        return "Inconclusive — models disagree"
    else:
        return "Moderate Risk"
'''

with open('../fusion.py', 'w') as f:
    f.write(fusion_code)

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix

# ROC data for drawing model
fpr_drawing, tpr_drawing, _ = roc_curve(y_test, y_prob)
roc_drawing = pd.DataFrame({'FPR': fpr_drawing, 'TPR': tpr_drawing})
roc_drawing.to_csv('../models/roc_drawing.csv', index=False)

# Confusion matrix for drawing
cm_drawing = confusion_matrix(y_test, y_pred)
cm_drawing_df = pd.DataFrame(cm_drawing,
                              index=['Actual Healthy', 'Actual PD'],
                              columns=['Predicted Healthy', 'Predicted PD'])
cm_drawing_df.to_csv('../models/confusion_matrix_drawing.csv')

Drawing metrics saved!
